# Lab 1: Graph basics, paths and centrality

**Data.** A synthetic business payment network: about 2,200 businesses, 1,500 individuals and a year of monthly payments (2025-09 to 2026-08). Everything is invented, and it has structure planted in it so we can check our answers later.

**You will:**
1. turn a payment table into a graph
2. see why a few hubs turn every picture into a hairball, and remove them
3. find shortest paths and measure how "small" the network is
4. rank businesses three ways (degree, betweenness, PageRank) and see where they disagree

Run each cell with Shift+Enter. Cells marked **Try it** invite you to change something.

In [ ]:
# Setup
import numpy as np, pandas as pd, networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import labkit as lk

nodes, edges = lk.load()
name = dict(zip(nodes.node_id, nodes.name))
sector = dict(zip(nodes.node_id, nodes.sector))
region = dict(zip(nodes.node_id, nodes.region))
display(pd.DataFrame({
    "table": ["nodes", "edges_monthly"],
    "rows": [len(nodes), len(edges)],
    "what one row is": ["an account holder (business O or individual P)", "payer -> payee total for one month"],
}))

## 1. A payment table is already a graph

Every row says *who paid whom, how much, in which month*. Read it as a graph:

* **node**: an account holder (a business or a person)
* **edge**: a payment relationship, pointing from payer to payee (**directed**)
* **weight**: dollars (or number of payments) on that edge
* **in-degree**: how many distinct payers a node has; **out-degree**: how many distinct payees
* **strength**: total dollars in or out (degree counts partners, strength counts money)

A customer x counterparty table is a **bipartite** graph: edges only run between the two kinds of node.

In [ ]:
# One business, as rows and as a picture
focus = "B0175"
rows = edges[(edges.src == focus) | (edges.dst == focus)]
year = rows.groupby(["src", "dst"], as_index=False)[["amount", "n_txn"]].sum()
year["direction"] = np.where(year.src == focus, "out (pays)", "in (is paid)")
year["counterparty"] = np.where(year.src == focus, year.dst, year.src).astype(object)
year["counterparty"] = [f"{c}  {name[c]}" for c in year.counterparty]

E = nx.DiGraph()
E.add_edges_from(zip(year.src, year.dst))
pos = lk.ego_layout(E, focus)                 # payers left, payees right
ecol = {(u, v): (lk.OUT_COLOR if u == focus else lk.IN_COLOR) for u, v in E.edges()}
fig = lk.draw(E, pos=pos, arrows=True, edge_colors=ecol, edge_width=1.5, height=520,
              color={n: (lk.INK if n == focus else lk.OTHER) for n in E},
              size={n: (22 if n == focus else 12) for n in E},
              labels={n: n for n in E}, hover={n: f"{n}: {name[n]}" for n in E},
              title=f"{focus} ({name[focus]}): blue arrows = money in, orange = money out", legend=False)
display(year[["direction", "counterparty", "amount", "n_txn"]].sort_values("direction").round(0), fig)

In [ ]:
# The whole year as one directed, weighted graph
G = lk.payment_graph(edges)                  # DiGraph: edge weight = log(1 + dollars), plus amount and n_txn
indeg, outdeg = dict(G.in_degree()), dict(G.out_degree())
display(pd.DataFrame({
    "measure": ["nodes", "edges (payer -> payee pairs)", "density (share of possible edges present)",
                f"{focus} in-degree", f"{focus} out-degree"],
    "value": [G.number_of_nodes(), G.number_of_edges(), f"{nx.density(G):.5f}", indeg[focus], outdeg[focus]],
}))

## 2. Hubs: the reason every payment network looks like a hairball

Real networks are not random. A few nodes are connected to almost everyone (a payroll processor, the tax agency, the utility). Compare the degree distribution with a random graph of the same size: the random graph has no hubs at all.

In [ ]:
# Degree distribution on log-log axes: payment network vs a random graph of the same size
U = G.to_undirected()
deg = pd.Series(dict(U.degree()))
R = nx.gnm_random_graph(U.number_of_nodes(), U.number_of_edges(), seed=1)
rdeg = pd.Series(dict(R.degree()))

fig = go.Figure()
for s, label, colr in ((deg, "payment network", lk.PALETTE[0]), (rdeg, "random graph, same nodes and edges", lk.PALETTE[1])):
    vc = s[s > 0].value_counts().sort_index()
    fig.add_trace(go.Scatter(x=vc.index, y=vc.values, mode="markers", name=label,
                             marker=dict(size=8, color=colr, line=dict(width=1, color=lk.SURFACE))))
fig.update_xaxes(type="log", title="degree (number of partners)")
fig.update_yaxes(type="log", title="number of nodes")
lk.style(fig, "A heavy tail: most nodes have a few partners, a handful have thousands", height=480)

In [ ]:
# Who are the nodes with the most partners?
top = deg.sort_values(ascending=False).head(10)
display(pd.DataFrame({"node": top.index, "name": [name[n] for n in top.index],
                      "sector": [sector[n] for n in top.index], "partners": top.values}))

These are **hubs**: payroll, tax, utility, insurance, card acquiring, fuel cards. They are real and important, but they connect almost everyone to almost everyone, so they hide the structure. Every network analysis starts by removing (or down-weighting) them.

A simple, robust rule: a node is a hub if its degree is more than 15 times the median degree among businesses. Percentile rules ("top 0.1%") break when hubs have similar degrees.

In [ ]:
# Hub rule: degree above 15 x median business degree
biz = [n for n in U if n.startswith("B")]
bdeg = deg[biz]
threshold = 15 * bdeg.median()
HUBS = sorted(bdeg[bdeg > threshold].index, key=lambda n: -deg[n])
display(pd.DataFrame({"hub": HUBS, "name": [name[h] for h in HUBS], "partners": [deg[h] for h in HUBS]}),
        f"median business degree {bdeg.median():.0f}, threshold {threshold:.0f}: {len(HUBS)} hubs")

**Discuss.** Is the big supplier at the bottom of that list a hub, or just a big supplier? Hub lists always need a human look. For the rest of the labs we remove the six payment-infrastructure hubs and keep the supplier.

In [ ]:
# Before and after: the same businesses, drawn with and without hubs (colour = region, which the bank knows)
HUBS = [h for h in HUBS if deg[h] > 400]          # the six payment-infrastructure hubs; the supplier stays
keep_regions = ["PA-West", "OH", "NJ", "KY"]
sample = [n for n in biz if region[n] in keep_regions and n not in HUBS]
cmap = {r: lk.PALETTE[i] for i, r in enumerate(keep_regions)}

fig = make_subplots(rows=1, cols=2, subplot_titles=("With the hubs", "Hubs removed"), horizontal_spacing=0.02)
for col, nodeset in ((1, sample + HUBS), (2, sample)):
    S = U.subgraph(nodeset)
    p = nx.spring_layout(S, seed=7, iterations=60)
    xs, ys = [], []
    for u, v in S.edges():
        xs += [p[u][0], p[v][0], None]
        ys += [p[u][1], p[v][1], None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(width=0.3, color="#cfcec7"),
                             hoverinfo="skip", showlegend=False), 1, col)
    for r in keep_regions:
        m = [n for n in S if region[n] == r]
        fig.add_trace(go.Scatter(x=[p[n][0] for n in m], y=[p[n][1] for n in m], mode="markers", name=r,
                                 marker=dict(size=5, color=cmap[r]), hovertext=[name[n] for n in m],
                                 hoverinfo="text", showlegend=(col == 2)), 1, col)
    h = [n for n in S if n in HUBS]
    if h:
        fig.add_trace(go.Scatter(x=[p[n][0] for n in h], y=[p[n][1] for n in h], mode="markers", name="hub",
                                 marker=dict(size=16, color=lk.INK), hovertext=[name[n] for n in h],
                                 hoverinfo="text"), 1, col)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
lk.style(fig, "Removing six hubs reveals the structure underneath", height=560)

## 3. Paths: how far apart are two businesses?

In 1956 Edsger Dijkstra designed the shortest-path algorithm in about twenty minutes on a café terrace in Amsterdam, with no pencil. Every navigation app is a descendant of it.

In a payment network a path answers questions like *how many hops is this business from a flagged entity?* or *through whom could money get from A to B?*

In [ ]:
# Shortest path between a Pittsburgh hospital supplier and an Iowa food processor
B = U.subgraph([n for n in U if n.startswith("B") and n not in HUBS]).copy()      # businesses, hubs removed
a = next(n for n in sorted(B) if region[n] == "PA-West" and sector[n] == "Health care")
b = next(n for n in sorted(B) if region[n] == "IN-IA" and sector[n] == "Food manufacturing")

path_no_hubs = nx.shortest_path(B, a, b)
path_with_hubs = nx.shortest_path(U, a, b)
show = lambda p: pd.DataFrame({"step": range(len(p)), "node": p, "name": [name[x] for x in p],
                               "region": [region[x] for x in p], "sector": [sector[x] for x in p]})
display(f"Hubs removed: {len(path_no_hubs) - 1} hops", show(path_no_hubs),
        f"With hubs: {len(path_with_hubs) - 1} hops (through a hub, which tells us nothing)", show(path_with_hubs))

In [ ]:
# Small world: distribution of shortest-path lengths (300 random starting points)
rng = np.random.default_rng(0)
def hop_counts(H, k=300):
    giant = H.subgraph(max(nx.connected_components(H), key=len))
    out = []
    for s in rng.choice(list(giant), k, replace=False):
        out += list(nx.single_source_shortest_path_length(giant, s).values())
    return pd.Series([x for x in out if x > 0])

with_h = hop_counts(U.subgraph([n for n in U if n.startswith("B")]))
without_h = hop_counts(B)
fig = go.Figure()
for s, label, c in ((with_h, f"with hubs (average {with_h.mean():.1f})", lk.PALETTE[1]),
                    (without_h, f"hubs removed (average {without_h.mean():.1f})", lk.PALETTE[0])):
    vc = s.value_counts(normalize=True).sort_index()
    fig.add_trace(go.Bar(x=vc.index, y=vc.values, name=label, marker=dict(color=c)))
fig.update_layout(barmode="group", xaxis_title="hops between two businesses", yaxis_title="share of pairs",
                  yaxis_tickformat=".0%")
lk.style(fig, "Six degrees? Facebook measured 3.57 people in between (4.57 hops) in 2016", height=430)

## 4. Centrality: who matters, and in what sense?

**Guess the winner.** Below is a small network. Before running the next cell, decide which node is the most important.

In [ ]:
# The guessing game
T = nx.Graph()
T.add_edges_from([("A", "B"), ("A", "C"), ("A", "D"), ("A", "E"), ("A", "L"), ("B", "C"), ("C", "D"),
                  ("H", "D"), ("H", "E"), ("H", "F"), ("H", "G"),
                  ("F", "G"), ("F", "I"), ("G", "I"), ("I", "J"), ("J", "K"), ("G", "J")])
tpos = nx.spring_layout(T, seed=11)
lk.draw(T, pos=tpos, labels={n: n for n in T}, size={n: 26 for n in T}, color={n: lk.OTHER for n in T},
        edge_width=2, title="Which node is the most important?", height=430, legend=False)

In [ ]:
# The reveal: three answers to "most important"
scores = pd.DataFrame({
    "degree (partners)": dict(T.degree()),
    "betweenness (sits on shortest paths)": nx.betweenness_centrality(T),
    "closeness (near everyone)": nx.closeness_centrality(T),
    "PageRank (linked to by important nodes)": nx.pagerank(T),
}).round(3)
fig = make_subplots(rows=1, cols=3, subplot_titles=("Degree", "Betweenness", "PageRank"), horizontal_spacing=0.02)
for i, col in enumerate(["degree (partners)", "betweenness (sits on shortest paths)",
                         "PageRank (linked to by important nodes)"]):
    s = scores[col] / scores[col].max()
    xs, ys = [], []
    for u, v in T.edges():
        xs += [tpos[u][0], tpos[v][0], None]
        ys += [tpos[u][1], tpos[v][1], None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(width=1.5, color=lk.EDGE),
                             hoverinfo="skip", showlegend=False), 1, i + 1)
    fig.add_trace(go.Scatter(x=[tpos[n][0] for n in T], y=[tpos[n][1] for n in T], mode="markers+text",
                             text=list(T), textposition="middle center", showlegend=False,
                             marker=dict(size=[12 + 34 * s[n] for n in T], color=lk.PALETTE[i]),
                             textfont=dict(color="white", size=11)), 1, i + 1)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
display(scores.sort_values("betweenness (sits on shortest paths)", ascending=False),
        lk.style(fig, "Same network, different winners: A has the most partners, H holds the two halves together", height=420))

**PageRank** (Brin and Page, 1998) imagines a random surfer who follows links and, 15% of the time, jumps to a random page. A page's score is the share of time the surfer spends there. You are important if important nodes point at you.

The founders reportedly tried to sell it to Excite in 1999 for 1 million dollars, then 750 thousand. Excite said no. Stanford, which held the patent, later sold its Google shares for 336 million dollars.

On a payment graph, "pointing at" means paying: a business scores high when it is paid by businesses that are themselves paid a lot.

In [ ]:
# PageRank by hand: power iteration in a few lines, then compare with networkx
P = nx.DiGraph([("A", "B"), ("A", "C"), ("B", "C"), ("C", "A"), ("D", "C"), ("E", "C"), ("E", "D")])
order = list(P)
ix = {n: i for i, n in enumerate(order)}
N, d = len(order), 0.85
M = np.zeros((N, N))
for u, v in P.edges():
    M[ix[v], ix[u]] = 1 / P.out_degree(u)          # column u spreads its score over u's payees
dangling = np.array([P.out_degree(n) == 0 for n in order])

r = np.full(N, 1 / N)
history = [r]
for _ in range(40):
    r = (1 - d) / N + d * (M @ r + r[dangling].sum() / N)
    history.append(r)
hist = pd.DataFrame(history, columns=order)

fig = go.Figure([go.Scatter(y=hist[c], mode="lines", name=c, line=dict(width=2, color=lk.PALETTE[i]))
                 for i, c in enumerate(order)])
fig.update_layout(xaxis_title="iteration", yaxis_title="score")
display(pd.DataFrame({"by hand": r.round(4), "networkx": pd.Series(nx.pagerank(P, alpha=d))[order].round(4).values},
                     index=order),
        lk.style(fig, "Power iteration settles in about 20 steps", height=380))

In [ ]:
# PageRank on the payment network: who sits where money ends up?
Bd = G.subgraph([n for n in G if n.startswith("B") and n not in HUBS])
pr = pd.Series(nx.pagerank(Bd, weight="amount"))
received = pd.Series({n: sum(d["amount"] for _, _, d in Bd.in_edges(n, data=True)) for n in Bd})
ranks = pd.DataFrame({"name": [name[n] for n in Bd], "sector": [sector[n] for n in Bd], "region": [region[n] for n in Bd],
                      "payers": [Bd.in_degree(n) for n in Bd], "dollars received": received.round(0),
                      "PageRank": pr}, index=list(Bd))
ranks["rank by payers"] = ranks.payers.rank(ascending=False, method="min").astype(int)
ranks["rank by dollars"] = ranks["dollars received"].rank(ascending=False, method="min").astype(int)
ranks["rank by PageRank"] = ranks.PageRank.rank(ascending=False, method="min").astype(int)
ranks.sort_values("PageRank", ascending=False).head(12)

**Try it.** Sort the table by `rank by payers` instead. Which businesses rank high on PageRank but not on payer count? Those are paid by businesses that are themselves central: suppliers to the suppliers.

In [ ]:
# Interactive: one regional network, sized by the measure you pick
R = U.subgraph([n for n in B if region[n] == "PA-West"]).copy()
rp = nx.spring_layout(R, seed=5, iterations=80)
measures = {"degree": pd.Series(dict(R.degree())), "betweenness": pd.Series(nx.betweenness_centrality(R)),
            "PageRank": pd.Series(nx.pagerank(R, weight="weight"))}
cm = lk.colour_map([sector[n] for n in R])
groups = {}
for n in R:
    groups.setdefault(sector[n] if sector[n] in cm else "Other sectors", []).append(n)
xs, ys = [], []
for u, v in R.edges():
    xs += [rp[u][0], rp[v][0], None]
    ys += [rp[u][1], rp[v][1], None]
fig = go.Figure([go.Scatter(x=xs, y=ys, mode="lines", line=dict(width=0.5, color=lk.EDGE), hoverinfo="skip",
                            showlegend=False)])
size_of = lambda m, members: [6 + 30 * measures[m][n] / measures[m].max() for n in members]
for gname, members in groups.items():
    fig.add_trace(go.Scatter(x=[rp[n][0] for n in members], y=[rp[n][1] for n in members], mode="markers",
                             name=gname, marker=dict(size=size_of("degree", members), color=cm.get(gname, lk.OTHER),
                                                     line=dict(width=1, color=lk.SURFACE)),
                             hovertext=[f"{name[n]}<br>{sector[n]}<br>degree {R.degree(n)}" for n in members],
                             hoverinfo="text"))
lk._layout(fig, "PA-West businesses (colour = sector): pick the size measure", height=660)
fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=1.0, xanchor="right", y=1.07,
                                    showactive=True, buttons=[
    dict(label=m, method="restyle", args=[{"marker.size": [size_of(m, g) for g in groups.values()]},
                                          list(range(1, len(groups) + 1))])
    for m in measures])])
fig

## Takeaways

* A payment table **is** a graph: nodes, directed weighted edges, degree and strength.
* Hubs hide structure. Remove them first, and review the list by hand.
* Shortest paths answer "how far" and "through whom", and hubs make every path meaningless.
* "Important" has several meanings: degree (many partners), betweenness (a bridge), PageRank (paid by the important).

Next: Lab 2 looks at the neighbourhood around a single customer.